In [1]:
# Cell 1 — Libraries + Data Load

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')

print("✅ Libraries imported!")

# Paths
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
DATA_PATH = os.path.join(BASE_DIR, 'data', 'paysim_india.csv')

# Data load karo — paysim_india.csv use karenge
# (model_features.csv nahi, kyunki usme nameOrig/nameDest
#  drop ho gaye the — graph ke liye yeh IDs chahiye)

print("\nLoading paysim_india.csv...")
df = pd.read_csv(DATA_PATH)

print(f"✅ Data loaded: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

print(f"\nUnique sender accounts (nameOrig): {df['nameOrig'].nunique():,}")
print(f"Unique receiver accounts (nameDest): {df['nameDest'].nunique():,}")

print(f"\nTransaction type distribution:")
print(df['txn_type_india'].value_counts())

print(f"\nFraud %: {df['isFraud'].mean()*100:.4f}%")
print(f"Any nulls: {df.isnull().sum().sum()}")

✅ Libraries imported!

Loading paysim_india.csv...
✅ Data loaded: (6362620, 24)

Columns: ['step', 'hour', 'day', 'day_of_week', 'is_weekend', 'is_odd_hour', 'type', 'txn_type_india', 'amount', 'amount_inr', 'nameOrig', 'oldbalance_inr', 'newbalance_inr', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'account_type', 'is_structuring', 'is_ctr_threshold', 'is_rtgs_range', 'balance_mismatch', 'has_mismatch', 'isFraud', 'isFlaggedFraud']

Unique sender accounts (nameOrig): 6,353,307
Unique receiver accounts (nameDest): 2,722,362

Transaction type distribution:
txn_type_india
ATM_WITHDRAWAL    2237500
UPI               2151495
CASH_DEPOSIT      1399284
NEFT               532909
NACH                41432
Name: count, dtype: int64

Fraud %: 0.1291%
Any nulls: 0


In [2]:
# Cell 2 — Graph-Relevant Subset + Degree Stats

print("=== FILTERING GRAPH-RELEVANT TRANSACTIONS ===\n")

# NEFT + ATM_WITHDRAWAL hi rakhenge —
# Inme nameDest ek genuine account hota hai (C prefix)
# UPI/CASH_DEPOSIT/NACH mein dest merchant ya system entity hota hai
# (PaySim ke original PAYMENT/CASH_IN type ki property)

graph_types = ['NEFT', 'ATM_WITHDRAWAL']

df_graph = df[df['txn_type_india'].isin(graph_types)].copy()

print(f"Total transactions: {len(df):,}")
print(f"Graph-relevant transactions (NEFT + ATM_WITHDRAWAL): {len(df_graph):,}")
print(f"Percentage kept: {len(df_graph)/len(df)*100:.2f}%")

print(f"\nBreakdown:")
print(df_graph['txn_type_india'].value_counts())

# Quick sanity check — nameDest prefix verify karo
# Expectation: NEFT/ATM_WITHDRAWAL mein dest 'C' se start hona chahiye
print(f"\n--- Sanity Check: nameDest prefix ---")
dest_prefix_check = df_graph['nameDest'].str[0].value_counts()
print(dest_prefix_check)

# Fraud check is subset mein
print(f"\nFraud in graph-relevant subset: {df_graph['isFraud'].sum():,}")
print(f"Fraud %: {df_graph['isFraud'].mean()*100:.4f}%")

# =============================
# Degree candidates nikalo
# =============================
# Out-degree = kitne UNIQUE accounts ko bheja (sender ke liye)
# In-degree  = kitne UNIQUE accounts se receive kiya (receiver ke liye)
# Yeh fan-out / fan-in ka raw signal hai

print("\n=== DEGREE STATISTICS (before building actual graph) ===")

out_degree = df_graph.groupby('nameOrig')['nameDest'].nunique()
in_degree = df_graph.groupby('nameDest')['nameOrig'].nunique()

print(f"\nOut-degree (fan-out candidates):")
print(out_degree.describe())
print(f"\nTop 5 senders by unique receivers (potential fan-out):")
print(out_degree.sort_values(ascending=False).head())

print(f"\nIn-degree (fan-in candidates):")
print(in_degree.describe())
print(f"\nTop 5 receivers by unique senders (potential fan-in / mule):")
print(in_degree.sort_values(ascending=False).head())

=== FILTERING GRAPH-RELEVANT TRANSACTIONS ===

Total transactions: 6,362,620
Graph-relevant transactions (NEFT + ATM_WITHDRAWAL): 2,770,409
Percentage kept: 43.54%

Breakdown:
txn_type_india
ATM_WITHDRAWAL    2237500
NEFT               532909
Name: count, dtype: int64

--- Sanity Check: nameDest prefix ---
nameDest
C    2770409
Name: count, dtype: int64

Fraud in graph-relevant subset: 8,213
Fraud %: 0.2965%

=== DEGREE STATISTICS (before building actual graph) ===

Out-degree (fan-out candidates):
count    2.768630e+06
mean     1.000643e+00
std      2.538327e-02
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      3.000000e+00
Name: nameDest, dtype: float64

Top 5 senders by unique receivers (potential fan-out):
nameOrig
C1902386530    3
C2098525306    3
C724452879     3
C1048374464    2
C129588043     2
Name: nameDest, dtype: int64

In-degree (fan-in candidates):
count    509565.000000
mean          5.436812
std           5.780369
min      

In [3]:
# Cell 3 — Fan-in Candidates: Fraud Cross-Check

print("=== TOP FAN-IN ACCOUNTS — FRAUD CROSS-CHECK ===\n")

# Top 20 fan-in accounts nikalo (in-degree se)
top_fanin = in_degree.sort_values(ascending=False).head(20)

results = []

for account, num_senders in top_fanin.items():
    # Is account ne kitna total amount receive kiya
    received_txns = df_graph[df_graph['nameDest'] == account]
    
    total_received = received_txns['amount_inr'].sum()
    fraud_count = received_txns['isFraud'].sum()
    txn_count = len(received_txns)
    account_type = received_txns['account_type'].iloc[0] if 'account_type' in received_txns.columns else 'unknown'
    
    results.append({
        'account': account,
        'unique_senders': num_senders,
        'total_txns': txn_count,
        'total_received_inr': total_received,
        'fraud_txn_count': fraud_count,
        'has_any_fraud': fraud_count > 0,
        'account_type': account_type
    })

fanin_summary = pd.DataFrame(results)

print(fanin_summary.to_string(index=False))

print(f"\n--- Summary ---")
print(f"Accounts with at least 1 fraud transaction: {fanin_summary['has_any_fraud'].sum()} / 20")
print(f"Total fraud transactions among top 20 fan-in accounts: {fanin_summary['fraud_txn_count'].sum()}")

# Compare — random accounts ka fraud rate kya hota hai (baseline)
print(f"\n--- Baseline Comparison ---")
print(f"Overall fraud rate in graph subset: {df_graph['isFraud'].mean()*100:.4f}%")

# High fan-in accounts ka combined fraud rate
high_fanin_accounts = top_fanin.index.tolist()
high_fanin_txns = df_graph[df_graph['nameDest'].isin(high_fanin_accounts)]
print(f"Fraud rate among top 20 fan-in accounts' transactions: {high_fanin_txns['isFraud'].mean()*100:.4f}%")

=== TOP FAN-IN ACCOUNTS — FRAUD CROSS-CHECK ===

    account  unique_senders  total_txns  total_received_inr  fraud_txn_count  has_any_fraud account_type
C1286084959              75          75          5927661.69                0          False      current
 C665576141              68          68          6854452.01                0          False      savings
C1360767589              68          68          2639278.16                0          False      savings
  C97730845              67          67         10471761.47                0          False      savings
 C248609774              64          64          2832008.92                0          False      current
C2006081398              63          63          8836621.21                0          False     jan_dhan
C2083562754              63          63          3951626.03                0          False      current
C1590550415              62          62          2983552.00                0          False      savings
C17895

In [4]:
# Cell 4 — Build the NetworkX Directed Graph

import time

print("=== BUILDING NETWORKX DIRECTED GRAPH ===\n")

start_time = time.time()

# DiGraph = Directed Graph
# Kyunki paisa ek direction mein flow karta hai (A → B, not B → A automatically)
G = nx.DiGraph()

# Edges add karenge row-by-row se nahi —
# nx.from_pandas_edgelist se ek shot mein, bahut fast hota hai
# source = nameOrig (sender), target = nameDest (receiver)
# edge_attr = wo columns jo har edge pe store karni hain

G = nx.from_pandas_edgelist(
    df_graph,
    source='nameOrig',
    target='nameDest',
    edge_attr=['amount_inr', 'isFraud', 'txn_type_india', 'hour', 'day'],
    create_using=nx.DiGraph()
)

elapsed = time.time() - start_time

print(f"✅ Graph built in {elapsed:.2f} seconds")
print(f"\nTotal nodes (accounts): {G.number_of_nodes():,}")
print(f"Total edges (transactions): {G.number_of_edges():,}")

# Edge count vs transaction count mismatch ho sakta hai —
# kyunki same nameOrig-nameDest pair multiple transactions kar sakta hai
# NetworkX DiGraph by default sirf EK edge rakhta hai per pair
# (multiple transactions ka data overwrite ho jaata hai!)
print(f"\nOriginal transaction rows: {len(df_graph):,}")
print(f"Graph edges: {G.number_of_edges():,}")
print(f"Difference (duplicate sender-receiver pairs collapsed): {len(df_graph) - G.number_of_edges():,}")

# Graph density — kitna 'connected' hai overall
print(f"\nGraph density: {nx.density(G):.8f}")

# Sample edge dekhte hain — attributes kaise stored hue
sample_edge = list(G.edges(data=True))[0]
print(f"\nSample edge structure:")
print(sample_edge)

# Memory check — bada graph hai, RAM kitna use ho raha
import sys
print(f"\nApprox graph object size check karne ke liye networkx info:")
print(nx.info(G)) if hasattr(nx, 'info') else print("(nx.info deprecated in newer versions, skip)")

=== BUILDING NETWORKX DIRECTED GRAPH ===

✅ Graph built in 21.40 seconds

Total nodes (accounts): 3,277,509
Total edges (transactions): 2,770,409

Original transaction rows: 2,770,409
Graph edges: 2,770,409
Difference (duplicate sender-receiver pairs collapsed): 0

Graph density: 0.00000026

Sample edge structure:
('C1305486145', 'C553264065', {'amount_inr': 15.39, 'isFraud': 1, 'txn_type_india': 'NEFT', 'hour': 1, 'day': 0})

Approx graph object size check karne ke liye networkx info:
(nx.info deprecated in newer versions, skip)


In [5]:
# Cell 5 — Round-Trip Detection Function (A → B → C → A)

print("=== ROUND-TRIP DETECTION (Cycles) ===\n")

def find_round_trips(G, max_length=4):
    """
    Round-trip cycles dhundta hai graph mein —
    A -> B -> C -> A jaise patterns, max_length hops tak.
    
    NetworkX ka simple_cycles() poore graph ke cycles deta hai,
    lekin bahut slow ho sakta hai large graphs pe agar
    length restriction na ho.
    
    length_bound parameter (NetworkX 3.x+) cycles ko 
    max_length tak hi restrict karta hai — yeh bahut zaroori hai
    warna 3.27M node graph pe yeh function kabhi khatam nahi hoga.
    """
    cycles = []
    try:
        # length_bound naye NetworkX versions mein available hai
        cycle_generator = nx.simple_cycles(G, length_bound=max_length)
        
        # Bahut bade graph mein cycles bahut zyada ho sakte hain —
        # safety ke liye ek cap lagate hain (pehle 50,000 dhundo, phir stop)
        count = 0
        cap = 50000
        for cycle in cycle_generator:
            if len(cycle) >= 3:  # 2-node cycle (A->B->A) ko exclude, min 3 chahiye
                cycles.append(cycle)
            count += 1
            if count >= cap:
                print(f"⚠️ Cap of {cap} reached, stopping search (graph bahut bada hai)")
                break
    except Exception as e:
        print(f"Error: {e}")
    
    return cycles

import time
start_time = time.time()

print("Searching for round-trip cycles (length 3-4)... yeh thoda time le sakta hai\n")
round_trips = find_round_trips(G, max_length=4)

elapsed = time.time() - start_time
print(f"\n✅ Search completed in {elapsed:.2f} seconds")
print(f"Total round-trip cycles found: {len(round_trips)}")

if len(round_trips) > 0:
    print(f"\nSample cycles (first 5):")
    for cycle in round_trips[:5]:
        print(cycle)
else:
    print("\n(Expected — humne predict kiya tha ki organic data mein round-trips rare/absent honge,")
    print(" kyunki har account almost ek-baar ka sender hai)")

=== ROUND-TRIP DETECTION (Cycles) ===

Searching for round-trip cycles (length 3-4)... yeh thoda time le sakta hai


✅ Search completed in 39.20 seconds
Total round-trip cycles found: 0

(Expected — humne predict kiya tha ki organic data mein round-trips rare/absent honge,
 kyunki har account almost ek-baar ka sender hai)


In [6]:
# Cell 6 — Fan-out & Fan-in Detection Function

print("=== FAN-OUT & FAN-IN DETECTION ===\n")

def detect_fan_pattern(G, df_graph, direction='out', degree_threshold=10, time_window_hours=24):
    """
    Fan-out: ek account -> bahut sare alag accounts (short time mein)
    Fan-in : bahut sare alag accounts -> ek account (short time mein)
    
    direction='out' -> fan-out detect karega (nameOrig group karega)
    direction='in'  -> fan-in detect karega (nameDest group karega)
    
    time_window_hours -> 'same time' ka criteria. PaySim mein
    'step' column = 1 hour ek unit hota hai, toh hum 'day' column
    use karenge clustering ke liye (zyada granular karna chahiye
    to 'hour' bhi add kar sakte hain, abhi day-level rakhte hain
    taaki bahut strict na ho jaaye).
    """
    
    if direction == 'out':
        group_col, target_col = 'nameOrig', 'nameDest'
    else:
        group_col, target_col = 'nameDest', 'nameOrig'
    
    # Har account + day combination ke liye unique counterparties count karo
    # Yeh 'same time window' criteria simulate karta hai
    grouped = df_graph.groupby([group_col, 'day'])[target_col].agg(
        unique_count='nunique',
        total_amount=('amount_inr', 'sum') if False else None  # placeholder, fix below
    )
    
    return grouped

# Upar wala function thoda buggy hai multi-agg ke saath, isliye clean version:

def detect_fan_pattern_v2(df_graph, direction='out', degree_threshold=10):
    """
    Clean version — har account ke liye, per-day unique counterparties
    aur total amount nikalta hai. Threshold se zyada wale flag hote hain.
    """
    if direction == 'out':
        group_col, target_col = 'nameOrig', 'nameDest'
    else:
        group_col, target_col = 'nameDest', 'nameOrig'
    
    grouped = df_graph.groupby([group_col, 'day']).agg(
        unique_counterparties=(target_col, 'nunique'),
        total_txns=(target_col, 'count'),
        total_amount=('amount_inr', 'sum'),
        fraud_txns=('isFraud', 'sum')
    ).reset_index()
    
    flagged = grouped[grouped['unique_counterparties'] >= degree_threshold].copy()
    flagged = flagged.sort_values('unique_counterparties', ascending=False)
    
    return grouped, flagged

# ===== FAN-OUT TEST =====
print("--- FAN-OUT (one account -> many, same day) ---")
all_out, flagged_out = detect_fan_pattern_v2(df_graph, direction='out', degree_threshold=5)
print(f"Threshold: >=5 unique receivers in a single day")
print(f"Accounts flagged: {len(flagged_out)}")
if len(flagged_out) > 0:
    print(flagged_out.head(10))
else:
    print("(Koi nahi mila — consistent with earlier finding: max out-degree overall tha sirf 3)")

# ===== FAN-IN TEST =====
print("\n--- FAN-IN (many -> one account, same day) ---")
all_in, flagged_in = detect_fan_pattern_v2(df_graph, direction='in', degree_threshold=10)
print(f"Threshold: >=10 unique senders in a single day")
print(f"Accounts flagged: {len(flagged_in)}")
if len(flagged_in) > 0:
    print(flagged_in.head(10))
else:
    print("(Koi nahi mila day-level clustering ke saath)")

print(f"\n--- Comparison: with vs without day-clustering ---")
print(f"Earlier (no time window) max in-degree: 75 unique senders OVERALL")
print(f"Now (single-day window) max in-degree: {all_in['unique_counterparties'].max()} unique senders in ONE DAY")

=== FAN-OUT & FAN-IN DETECTION ===

--- FAN-OUT (one account -> many, same day) ---
Threshold: >=5 unique receivers in a single day
Accounts flagged: 0
(Koi nahi mila — consistent with earlier finding: max out-degree overall tha sirf 3)

--- FAN-IN (many -> one account, same day) ---
Threshold: >=10 unique senders in a single day
Accounts flagged: 7753
            nameDest  day  unique_counterparties  total_txns  total_amount  \
275512   C1286084959    0                     58          58    2146790.02   
1135597   C248609774    0                     56          56    2633909.04   
347299   C1360767589    0                     55          55    2396156.72   
1041149  C2083562754    0                     55          55    2515777.33   
1191097   C306206744    0                     52          52    1626486.20   
1533611   C665576141    0                     51          51    2700069.59   
966321   C2006081398    0                     50          50    1961871.65   
1843143   C985934102 

In [7]:
# Cell 7 — Day-0 Investigation + Account Type Cross-Check

print("=== INVESTIGATING DAY=0 ANOMALY ===\n")

# Step 1: Har din ka transaction count dekho —
# kya day=0 genuinely outlier hai ya normal hai?
daily_txn_count = df_graph.groupby('day').size()

print("Transactions per day (first 10 days):")
print(daily_txn_count.head(10))

print(f"\nDay 0 transaction count: {daily_txn_count.iloc[0]:,}")
print(f"Average transactions per day (all days): {daily_txn_count.mean():,.0f}")
print(f"Day 0 vs average ratio: {daily_txn_count.iloc[0] / daily_txn_count.mean():.2f}x")

print(f"\nTotal unique days in dataset: {df_graph['day'].nunique()}")
print(f"Day range: {df_graph['day'].min()} to {df_graph['day'].max()}")

# Step 2: Account type cross-check for flagged fan-in accounts
print("\n\n=== ACCOUNT TYPE BREAKDOWN — FLAGGED FAN-IN ACCOUNTS ===\n")

# nameDest ke against account_type chahiye —
# lekin account_type df mein sender (nameOrig) side se associated hai ya generic hai?
# pehle check karte hain account_type kis column se related hai
print("Sample rows with account_type:")
print(df_graph[['nameOrig', 'nameDest', 'account_type']].head(3))

# flagged_in mein se account_type laate hain
# (nameDest ko df_graph mein nameOrig se match karna padega agar account_type
#  sender-side attribute hai, ya phir directly nameDest row se agar yeh
#  receiver ka bhi describe karta hai)

# Pehle dest-side account ka type dhundo (uska apna account_type, jab woh sender bana ho kabhi)
dest_account_types = df_graph.groupby('nameDest')['account_type'].first()  # placeholder approach

flagged_in_with_type = flagged_in.merge(
    dest_account_types.rename('dest_account_type'),
    left_on='nameDest',
    right_index=True,
    how='left'
)

print(f"\nAccount type distribution among {len(flagged_in_with_type)} flagged fan-in accounts:")
print(flagged_in_with_type['dest_account_type'].value_counts())
print(f"\nMissing account_type (account never appeared as sender): {flagged_in_with_type['dest_account_type'].isna().sum()}")

# Compare with overall account_type distribution (baseline)
print(f"\n--- Baseline: overall account_type distribution (as senders) ---")
print(df_graph['account_type'].value_counts())


=== INVESTIGATING DAY=0 ANOMALY ===

Transactions per day (first 10 days):
day
0    250180
1    202388
2      1804
3      7145
4      3586
5    195729
6    185960
7    196933
8    181672
9    171785
dtype: int64

Day 0 transaction count: 250,180
Average transactions per day (all days): 89,368
Day 0 vs average ratio: 2.80x

Total unique days in dataset: 31
Day range: 0 to 30


=== ACCOUNT TYPE BREAKDOWN — FLAGGED FAN-IN ACCOUNTS ===

Sample rows with account_type:
       nameOrig    nameDest account_type
2   C1305486145  C553264065      current
3    C840083671   C38997010      current
15   C905080434  C476402209      savings

Account type distribution among 7753 flagged fan-in accounts:
dest_account_type
savings     3844
current     2005
jan_dhan     783
student      729
business     392
Name: count, dtype: int64

Missing account_type (account never appeared as sender): 0

--- Baseline: overall account_type distribution (as senders) ---
account_type
savings     1385056
current      6925

In [8]:
# Cell 8 (Revised) — Faker Setup + Round-Trip Injection (Fraud + Benign)

from faker import Faker
import random

print("=== FAKER SETUP + ROUND-TRIP INJECTION (FRAUD + BENIGN) ===\n")

fake = Faker('en_IN')
Faker.seed(42)
random.seed(42)
np.random.seed(42)

print("✅ Faker initialized with seed=42 (reproducible)")

all_real_accounts = pd.unique(df_graph[['nameOrig', 'nameDest']].values.ravel())
print(f"Total unique real accounts available to reuse: {len(all_real_accounts):,}")

def generate_round_trip_chains(all_accounts, n_chains, chain_length=3, is_fraud=True, pattern_prefix='RT'):
    """
    Round-trip chains banata hai — fraud ya benign, dono ke liye same function.
    
    Fraud version —
    - Odd hours (1,2,3,23) — raat ko fraud activity ka classic signal
    - Amount decay har hop pe (commission/fees cut, laundering ka sign)
    - High starting amount (₹20k - ₹2L) — bada paisa launder karne ki koshish
    
    Benign version —
    - Normal business hours (9AM - 6PM)
    - Amount roughly consistent (minor variation, jaise business settlement/refund)
    - Moderate amount (₹2k - ₹50k) — typical business transaction range
    
    is_fraud flag se hum decide karte hain kaunsa variant banana hai.
    """
    synthetic_rows = []
    
    for i in range(n_chains):
        chain_accounts = random.sample(list(all_accounts), chain_length)
        chain_accounts.append(chain_accounts[0])  # round-trip complete karo
        
        base_day = random.randint(0, 30)
        
        if is_fraud:
            base_amount = random.uniform(20000, 200000)
            base_hour = random.choice([1, 2, 3, 23])
            decay_rate = 0.90  # 10% cut har hop pe
            account_tag = 'synthetic_roundtrip_fraud'
        else:
            base_amount = random.uniform(2000, 50000)
            base_hour = random.choice(range(9, 18))  # business hours 9AM-6PM
            decay_rate = 0.98  # bahut kam variation, almost same amount wapas
            account_tag = 'synthetic_roundtrip_benign'
        
        for hop in range(len(chain_accounts) - 1):
            sender = chain_accounts[hop]
            receiver = chain_accounts[hop + 1]
            amount_this_hop = base_amount * (decay_rate ** hop)
            
            synthetic_rows.append({
                'nameOrig': sender,
                'nameDest': receiver,
                'amount_inr': round(amount_this_hop, 2),
                'isFraud': 1 if is_fraud else 0,
                'txn_type_india': 'NEFT',
                'hour': base_hour,
                'day': base_day,
                'account_type': account_tag,
                'pattern_id': f'{pattern_prefix}_{i:04d}'
            })
    
    return synthetic_rows

# ===== FRAUD round-trips — test batch: 50 chains =====
synthetic_rt_fraud = generate_round_trip_chains(
    all_real_accounts, n_chains=50, chain_length=3, is_fraud=True, pattern_prefix='RT_FRAUD'
)

# ===== BENIGN round-trips — comparison batch: 20 chains =====
synthetic_rt_benign = generate_round_trip_chains(
    all_real_accounts, n_chains=20, chain_length=3, is_fraud=False, pattern_prefix='RT_BENIGN'
)

df_synthetic_rt_fraud = pd.DataFrame(synthetic_rt_fraud)
df_synthetic_rt_benign = pd.DataFrame(synthetic_rt_benign)

print(f"\n✅ Fraud round-trips generated: {len(df_synthetic_rt_fraud)} rows ({50} chains × 3 hops)")
print(f"✅ Benign round-trips generated: {len(df_synthetic_rt_benign)} rows ({20} chains × 3 hops)")

print(f"\n--- Sample FRAUD chain (RT_FRAUD_0000) ---")
print(df_synthetic_rt_fraud[df_synthetic_rt_fraud['pattern_id']=='RT_FRAUD_0000'][['nameOrig','nameDest','amount_inr','hour','isFraud']])

print(f"\n--- Sample BENIGN chain (RT_BENIGN_0000) ---")
print(df_synthetic_rt_benign[df_synthetic_rt_benign['pattern_id']=='RT_BENIGN_0000'][['nameOrig','nameDest','amount_inr','hour','isFraud']])

print(f"\n--- Key differences ---")
print(f"Fraud avg amount: ₹{df_synthetic_rt_fraud['amount_inr'].mean():,.2f} | Benign avg amount: ₹{df_synthetic_rt_benign['amount_inr'].mean():,.2f}")
print(f"Fraud hours used: {sorted(df_synthetic_rt_fraud['hour'].unique())} | Benign hours used: {sorted(df_synthetic_rt_benign['hour'].unique())}")


=== FAKER SETUP + ROUND-TRIP INJECTION (FRAUD + BENIGN) ===

✅ Faker initialized with seed=42 (reproducible)
Total unique real accounts available to reuse: 3,277,509

✅ Fraud round-trips generated: 150 rows (50 chains × 3 hops)
✅ Benign round-trips generated: 60 rows (20 chains × 3 hops)

--- Sample FRAUD chain (RT_FRAUD_0000) ---
      nameOrig     nameDest  amount_inr  hour  isFraud
0   C141374717  C2038922594    69505.28     2        1
1  C2038922594    C92716749    62554.75     2        1
2    C92716749   C141374717    56299.27     2        1

--- Sample BENIGN chain (RT_BENIGN_0000) ---
      nameOrig     nameDest  amount_inr  hour  isFraud
0  C1849017865  C1706768211    30040.52    17        0
1  C1706768211   C944012834    29439.71    17        0
2   C944012834  C1849017865    28850.92    17        0

--- Key differences ---
Fraud avg amount: ₹94,239.08 | Benign avg amount: ₹26,513.72
Fraud hours used: [np.int64(1), np.int64(2), np.int64(3), np.int64(23)] | Benign hours used: [n